In [1]:
from TrappedAtomsSimulation.force_calculation import (
    pair_torch_simple_fp,
    pair_torch_chunk_fp,
    pair_triton_fp_v1,
    pair_triton_fp_v2,
    pair_keops_fp
)
import torch
import time

In [4]:
device = torch.device('cuda')
N = 10000 
r = .01
c = .1
N_REPEATS_GPU = 10


positions = torch.rand(N, 3, device=device, dtype=torch.float32)

print(f"--- Benchmark (N={N}, D=3, {N_REPEATS_GPU} Wiederholungen) ---")

# ========================================================================
# --- Benchmark ---
# ========================================================================

# --- Benchmark Torch Simplw ---

avg_simple_ms = 0.0
pot_simple = torch.tensor(0.0)
forces_simple = torch.zeros_like(positions)

if N <= 10000:
    print("Benchmark Torch Simple")
    pair_torch_simple_fp(positions, r, c) # Warmup
    torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(N_REPEATS_GPU):
        forces_simple, pot_simple = pair_torch_simple_fp(positions, r, c)
    torch.cuda.synchronize()
    end = time.perf_counter()
    avg_simple_ms = ((end - start) * 1000) / N_REPEATS_GPU
else:
    print(f"Benchmark Torch Simple Übersprungen (N={N} > 10000)")

# --- Benchmark Torch Chunked ---
print("Benchmark Torch Chunked")
pair_torch_chunk_fp(positions, r, c) # Warmup
torch.cuda.synchronize()

start = time.perf_counter()
for _ in range(N_REPEATS_GPU):
    forces_chunk, pot_chunk = pair_torch_chunk_fp(positions, r, c)
torch.cuda.synchronize()
end = time.perf_counter()
avg_chunk_ms = ((end - start) * 1000) / N_REPEATS_GPU

# --- Benchmark V1 (Importiert) ---
print("Benchmark Triton-Kernel v1")
pair_triton_fp_v1(positions, r, c) # Warmup
torch.cuda.synchronize()

start = time.perf_counter()
for _ in range(N_REPEATS_GPU):
    forces_v1, pot_v1 = pair_triton_fp_v1(positions, r, c)
torch.cuda.synchronize()
end = time.perf_counter()
avg_v1_ms = ((end - start) * 1000) / N_REPEATS_GPU

# --- Benchmark V2  ---
print("Benchmark Triton-Kernel v2 ")
pair_triton_fp_v2(positions, r, c) # Warmup
torch.cuda.synchronize()

start = time.perf_counter()
for _ in range(N_REPEATS_GPU):
    forces_v2, pot_v2 = pair_triton_fp_v2(positions, r, c)
torch.cuda.synchronize()
end = time.perf_counter()
avg_v2_ms = ((end - start) * 1000) / N_REPEATS_GPU

# --- Benchmark KeOps ---
print("Benchmark KeOps")
pair_keops_fp(positions, r, c)
torch.cuda.synchronize()

start = time.perf_counter()
for _ in range(N_REPEATS_GPU):
    forces_keops, pot_keops = pair_keops_fp(positions, r, c)
torch.cuda.synchronize()
end = time.perf_counter()
avg_keops_ms = ((end - start) * 1000) / N_REPEATS_GPU

print("\nBenchmark abgeschlossen.")


# --- Ergebnisse ---
print(f"\n--- Benchmark-Ergebnisse (N={N}, D=3) ---")
print(f"Durchschnittliche Zeit über {N_REPEATS_GPU} Wiederholungen (in ms):")


if N <= 10000:
    print(f"  Torch Simple:  {avg_simple_ms:>10.4f} ms")
else:
    print(f"  Torch Simple:    (Übersprungen)")

print(f"  Torch Chunked: {avg_chunk_ms:>10.4f} ms")
print(f"  Triton v1:     {avg_v1_ms:>10.4f} ms")
print(f"  Triton v2:     {avg_v2_ms:>10.4f} ms ")
print(f"  KeOps:         {avg_keops_ms:>10.4f} ms")


print("\nValidierung (Potenziale):")
if N <= 10000:
    print(f"  Pot Simple:      {pot_simple.item():.6f}")
print(f"  Pot Chunk:       {pot_chunk.item():.6f}")
print(f"  Pot V1:   \t \t {pot_v1.item():.6f}")
print(f"  Pot V2:   \t\t \t {pot_v2.item():.6f}")
print(f"  Pot KeOps:       {pot_keops.item():.6f}")


--- Benchmark (N=10000, D=3, 10 Wiederholungen) ---
Benchmark Torch Simple
Benchmark Torch Chunked
Benchmark Triton-Kernel v1
Benchmark Triton-Kernel v2 
Benchmark KeOps

Benchmark abgeschlossen.

--- Benchmark-Ergebnisse (N=10000, D=3) ---
Durchschnittliche Zeit über 10 Wiederholungen (in ms):
  Torch Simple:     21.5885 ms
  Torch Chunked:     6.1066 ms
  Triton v1:         0.4037 ms
  Triton v2:         0.4949 ms 
  KeOps:             1.3704 ms

Validierung (Potenziale):
  Pot Simple:      73.492798
  Pot Chunk:       73.492798
  Pot V1:   	 	 73.492783
  Pot V2:   		 	 73.492783
  Pot KeOps:       73.492798
